In [7]:
library(ggplot2)
library(dplyr)
library(patchwork)

# =========================
# 1. 路径设置
# =========================
RES_OUT <- "/mnt/zhangzheng_group/liuz-52/Test_R/01_Figure1_python/inferCNV_results"

meta_file <- file.path(
  RES_OUT,
  "Figure1_marker_malignant_inferCNV_intersection_metadata.csv"
)

meta <- read.csv(
  meta_file,
  row.names = 1,
  check.names = FALSE
)

# =========================
# 2. 重新定义简洁分组名称
# =========================
meta$malignant_inferCNV_simple <- dplyr::case_when(
  meta$malignant_inferCNV == "Non-malignant" ~ "Non-malignant",
  meta$malignant_inferCNV == "Marker-defined malignant only" ~ "Marker-only",
  meta$malignant_inferCNV == "inferCNV-positive non-marker cell" ~ "inferCNV-only",
  meta$malignant_inferCNV == "High-confidence malignant" ~ "High-confidence",
  TRUE ~ NA_character_
)

meta$malignant_inferCNV_simple <- factor(
  meta$malignant_inferCNV_simple,
  levels = c(
    "Non-malignant",
    "Marker-only",
    "inferCNV-only",
    "High-confidence"
  )
)

# =========================
# 3. 配色
# =========================
group_cols <- c(
  "Non-malignant" = "#4DAF4A",
  "Marker-only" = "#377EB8",
  "inferCNV-only" = "#E41A1C",
  "High-confidence" = "#FF7F00"
)

# =========================
# 4. Fig. S3B: 百分比柱状图
# =========================
df_count <- meta %>%
  filter(!is.na(malignant_inferCNV_simple)) %>%
  count(malignant_inferCNV_simple) %>%
  mutate(
    percent = n / sum(n) * 100,
    label = paste0(sprintf("%.1f", percent), "%\n", "n=", n)
  )

p_bar <- ggplot(
  df_count,
  aes(
    x = malignant_inferCNV_simple,
    y = percent,
    fill = malignant_inferCNV_simple
  )
) +
  geom_col(
    width = 0.7,
    color = "black",
    linewidth = 0.3
  ) +
  geom_text(
    aes(label = label),
    vjust = -0.25,
    size = 3.6
  ) +
  scale_fill_manual(values = group_cols) +
  scale_y_continuous(
    limits = c(0, max(df_count$percent) * 1.25),
    expand = c(0, 0)
  ) +
  labs(
    x = NULL,
    y = "Percentage of cells (%)"
  ) +
  theme_classic(base_size = 12) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(
      angle = 35,
      hjust = 1,
      vjust = 1,
      color = "black"
    ),
    axis.text.y = element_text(color = "black"),
    axis.title.y = element_text(color = "black"),
    plot.margin = margin(10, 15, 10, 10)
  )

# =========================
# 5. Fig. S3C: inferCNV score violin
# =========================
p_violin <- ggplot(
  meta,
  aes(
    x = malignant_inferCNV_simple,
    y = inferCNV_score,
    fill = malignant_inferCNV_simple
  )
) +
  geom_violin(
    scale = "width",
    trim = TRUE,
    color = "black",
    linewidth = 0.25,
    na.rm = TRUE
  ) +
  geom_boxplot(
    width = 0.15,
    outlier.size = 0.2,
    color = "black",
    linewidth = 0.3,
    na.rm = TRUE
  ) +
  scale_fill_manual(values = group_cols) +
  labs(
    x = NULL,
    y = "inferCNV score"
  ) +
  theme_classic(base_size = 12) +
  theme(
    legend.position = "none",
    axis.text.x = element_text(
      angle = 35,
      hjust = 1,
      vjust = 1,
      color = "black"
    ),
    axis.text.y = element_text(color = "black"),
    axis.title.y = element_text(color = "black"),
    plot.margin = margin(10, 10, 10, 15)
  )

# =========================
# 6. 合并 B 和 C
# =========================
p_supp3_BC <- p_bar + p_violin +
  plot_layout(
    ncol = 2,
    widths = c(1, 1)
  ) +
  plot_annotation(
    tag_levels = list(c("B", "C"))
  ) &
  theme(
    plot.tag = element_text(
      size = 16,
      face = "bold"
    )
  )

# =========================
# 7. 保存图片
# =========================
ggsave(
  filename = file.path(
    RES_OUT,
    "SupplementaryFig3_BC_marker_inferCNV_overlap_percentage.pdf"
  ),
  plot = p_supp3_BC,
  width = 10,
  height = 4.5
)

ggsave(
  filename = file.path(
    RES_OUT,
    "SupplementaryFig3_BC_marker_inferCNV_overlap_percentage.png"
  ),
  plot = p_supp3_BC,
  width = 10,
  height = 4.5,
  dpi = 300
)

# =========================
# 8. 输出统计表
# =========================
write.csv(
  df_count,
  file = file.path(
    RES_OUT,
    "SupplementaryFig3_marker_inferCNV_classification_summary.csv"
  ),
  row.names = FALSE,
  quote = FALSE
)

print(df_count)

cat("Done. Files saved to:\n")
cat(RES_OUT, "\n")

  malignant_inferCNV_simple     n   percent          label
1             Non-malignant 25219 41.529164 41.5%\nn=25219
2               Marker-only  5069  8.347331   8.3%\nn=5069
3             inferCNV-only 21430 35.289662 35.3%\nn=21430
4           High-confidence  9008 14.833844  14.8%\nn=9008
Done. Files saved to:
/mnt/zhangzheng_group/liuz-52/Test_R/01_Figure1_python/inferCNV_results 
